# Transcribe Pipeline — Colab GPU worker

Канонический Colab worker для распределённого pipeline: Google Drive exchange → ASR → diarization → ECAPA embeddings → exchange results.

Оркестрация и состояние остаются на Oracle/PostgreSQL. Notebook обрабатывает опубликованные exchange jobs и не обходит lease/exchange слой.


In [ ]:
import subprocess
from pathlib import Path

repo = Path('/content/transcribe')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/JoTalbot/transcribe.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)

subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-colab.txt')], check=True)
print('Repository:', repo)

from google.colab import drive, userdata
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required. Select Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0))

try:
    HF_TOKEN = userdata.get('HUGGINGFACE_TOKEN')
except Exception as exc:
    raise RuntimeError('Create the Colab Secret HUGGINGFACE_TOKEN and allow notebook access.') from exc
if not HF_TOKEN:
    raise RuntimeError('HUGGINGFACE_TOKEN is empty.')

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/transcribe')
for p in (ROOT / 'exchange', ROOT / 'exchange' / 'artifacts', ROOT / 'input'):
    p.mkdir(parents=True, exist_ok=True)


## Запуск canonical exchange worker

Worker загружает Whisper large-v3, pyannote speaker-diarization-3.1 и SpeechBrain ECAPA на GPU. Он забирает jobs из `exchange/requests`, публикует результаты в `exchange/results` и физические artifacts в `exchange/artifacts`.

`--once` предназначен для ручного smoke/E2E прогона. Для постоянного worker можно убрать `--once`, но Colab не гарантирует бессрочную сессию.


In [ ]:
%cd /content/transcribe
!python scripts/colab_exchange_worker.py --root /content/drive/MyDrive/transcribe/exchange --input /content/drive/MyDrive/transcribe/input --output /content/drive/MyDrive/transcribe/exchange/artifacts --once


## После выполнения

Oracle должен синхронизировать `exchange/results` и `exchange/artifacts` обратно, применить результат через PostgreSQL lease ownership и продолжить следующий stage. Не редактируйте `processing` вручную.


In [ ]:
from pathlib import Path
exchange = Path('/content/drive/MyDrive/transcribe/exchange')
for name in ('requests', 'processing', 'results', 'artifacts'):
    p = exchange / name
    count = len(list(p.glob('*'))) if p.exists() else 0
    print(f'{name}: {count} entries')
